# Incident Response Runbook: Lateral Movement - Internal Spearphishing

**Tactic:** Lateral Movement
**Technique:** Internal Spearphishing (T1534)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Internal Spearphishing activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime, timedelta
import sys
import os

# Add parent directory to path for imports
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_data_collector import CrowdStrikeDataCollector
from misp.misp_integration import MISPIntegration
from iris.iris_integration import IRISIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Step 1: Initial Alert Analysis & Detection
print("=" * 60)
print("STEP 1: Initial Alert Analysis & Detection")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeDataCollector()
misp = MISPIntegration()
iris = IRISIntegration()
shuffle = ShuffleIntegration()

# Create incident case in IRIS
incident_data = {
    'title': 'Internal Spearphishing Attack - T1534',
    'description': 'Potential internal spearphishing campaign detected - suspicious internal communications and lateral movement identified',
    'severity': 'HIGH',
    'category': 'Insider Threat',
    'tags': ['spearphishing', 'internal', 'lateral-movement', 'insider-threat', 't1534']
}

case_id = iris.create_case(incident_data)
print(f" Created IRIS case: {case_id}")

# Detection queries for internal spearphishing indicators
print("\n Executing internal spearphishing detection queries...")

# Splunk query for suspicious internal emails
internal_email_query = """
index=* sourcetype=email
| where sender_domain == recipient_domain
| where (attachment_name =~ "*.exe" OR attachment_name =~ "*.scr" OR attachment_name =~ "*.pif" OR attachment_name =~ "*.bat")
OR (subject =~ "(?i)(urgent|important|confidential|password|credentials|login)")
OR (body =~ "(?i)(click here|download|open attachment|verify account)")
| stats count by sender, recipient, subject, attachment_name
| where count > 1
"""

email_results = splunk.execute_query(internal_email_query)
print(f" Found {len(email_results)} suspicious internal emails")

# Splunk query for unusual login patterns
login_anomaly_query = """
index=* sourcetype=authentication
| where action="success"
| stats count by user, src_ip, dest_host
| where count > 10 AND dest_host != src_ip
| sort -count
"""

login_results = splunk.execute_query(login_anomaly_query)
print(f" Found {len(login_results)} unusual login patterns")

# Splunk query for lateral movement file access
file_access_query = """
index=* sourcetype=file_access
| where user != file_owner
| where action="access" OR action="modify"
| stats count by user, file_path, file_owner
| where count > 5
| sort -count
"""

file_results = splunk.execute_query(file_access_query)
print(f" Found {len(file_results)} suspicious file access patterns")

# CrowdStrike query for suspicious process behavior
cs_query = """
event_simpleName=ProcessRollup2
| where (ImageFileName =~ "*powershell*" OR ImageFileName =~ "*cmd*" OR ImageFileName =~ "*rundll32*")
| where CommandLine =~ "*download*" OR CommandLine =~ "*invoke*" OR CommandLine =~ "*webclient*"
| stats count by ComputerName, UserName, ImageFileName, CommandLine
"""

cs_results = crowdstrike.execute_query(cs_query)
print(f" Found {len(cs_results)} suspicious process activities")

# Check for known spearphishing IOCs in MISP
spearphishing_indicators = misp.search_indicators("spearphishing", limit=50)
print(f" Retrieved {len(spearphishing_indicators)} spearphishing indicators from MISP")

# Compile detection results
detection_summary = {
    'case_id': case_id,
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Lateral Movement',
    'technique': 'Internal Spearphishing (T1534)',
    'severity': 'HIGH',
    'indicators': {
        'suspicious_emails': len(email_results),
        'login_anomalies': len(login_results),
        'file_access_patterns': len(file_results),
        'suspicious_processes': len(cs_results),
        'misp_indicators': len(spearphishing_indicators)
    },
    'affected_users': list(set([r.get('recipient', r.get('user', 'unknown')) 
                               for r in email_results + login_results + file_results])),
    'compromised_accounts': list(set([r.get('user', 'unknown') for r in login_results]))
}

print(f"\n📊 Detection Summary:")
print(f"  Case ID: {detection_summary['case_id']}")
print(f"  Severity: {detection_summary['severity']}")
print(f"  Affected Users: {len(detection_summary['affected_users'])}")
print(f"  Compromised Accounts: {len(detection_summary['compromised_accounts'])}")
print(f"  Suspicious Emails: {detection_summary['indicators']['suspicious_emails']}")
print(f"  Login Anomalies: {detection_summary['indicators']['login_anomalies']}")
print(f"  File Access Patterns: {detection_summary['indicators']['file_access_patterns']}")

# Trigger Shuffle workflow for automated response
if detection_summary['indicators']['suspicious_emails'] > 0 or detection_summary['indicators']['login_anomalies'] > 0:
    shuffle.trigger_workflow('internal_spearphishing_response', detection_summary)
    print(" Triggered automated internal spearphishing response workflow")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment Actions
print("\n" + "=" * 60)
print("STEP 2: Containment Actions")
print("=" * 60)

containment_actions = []

# Quarantine suspicious emails
print("\n📧 Quarantining suspicious emails...")
for email in email_results:
    try:
        quarantine_result = shuffle.quarantine_email(email['sender'], email['recipient'], email['subject'])
        if quarantine_result.get('success'):
            containment_actions.append(f" Quarantined email: {email['subject']} from {email['sender']}")
            print(f"   Quarantined email: {email['subject']} from {email['sender']}")
    except Exception as e:
        containment_actions.append(f" Failed to quarantine email from {email['sender']}: {str(e)}")
        print(f"   Failed to quarantine email from {email['sender']}: {str(e)}")

# Disable compromised accounts
print("\n👤 Disabling compromised accounts...")
for account in detection_summary['compromised_accounts']:
    try:
        disable_result = shuffle.disable_account(account)
        if disable_result.get('success'):
            containment_actions.append(f" Disabled account: {account}")
            print(f"   Disabled account: {account}")
    except Exception as e:
        containment_actions.append(f" Failed to disable account {account}: {str(e)}")
        print(f"   Failed to disable account {account}: {str(e)}")

# Block suspicious internal communications
print("\n Blocking suspicious internal communications...")
communication_query = """
index=* sourcetype=email OR sourcetype=chat
| where sender IN (detection_summary['compromised_accounts'])
| stats count by sender, recipient
| where count > 3
"""

communication_results = splunk.execute_query(communication_query)
for comm in communication_results:
    try:
        block_result = shuffle.block_internal_communication(comm['sender'], comm['recipient'])
        if block_result.get('success'):
            containment_actions.append(f" Blocked communication: {comm['sender']} → {comm['recipient']}")
            print(f"   Blocked communication: {comm['sender']} → {comm['recipient']}")
    except Exception as e:
        containment_actions.append(f" Failed to block communication {comm['sender']} → {comm['recipient']}: {str(e)}")
        print(f"   Failed to block communication {comm['sender']} → {comm['recipient']}: {str(e)}")

# Isolate affected endpoints via CrowdStrike
print("\n Isolating affected endpoints...")
for user in detection_summary['affected_users']:
    # Find endpoints associated with user
    endpoint_query = f"""
    index=* sourcetype=authentication
    | where user="{user}"
    | stats latest(src_ip) as endpoint by user
    """
    endpoint_results = splunk.execute_query(endpoint_query)
    for endpoint in endpoint_results:
        try:
            isolation_result = crowdstrike.isolate_host(endpoint['endpoint'])
            if isolation_result.get('success'):
                containment_actions.append(f" Isolated endpoint: {endpoint['endpoint']} ({user})")
                print(f"   Isolated endpoint: {endpoint['endpoint']} ({user})")
        except Exception as e:
            containment_actions.append(f" Failed to isolate endpoint {endpoint['endpoint']}: {str(e)}")
            print(f"   Failed to isolate endpoint {endpoint['endpoint']}: {str(e)}")

# Enable enhanced monitoring for affected users
print("\n📊 Enabling enhanced monitoring...")
try:
    monitoring_config = {
        'users': detection_summary['affected_users'],
        'monitor_email_activity': True,
        'monitor_file_access': True,
        'monitor_authentication': True,
        'alert_sensitivity': 'high'
    }
    splunk.setup_user_monitoring(monitoring_config)
    containment_actions.append(" Enhanced user monitoring enabled")
    print("   Enhanced user monitoring enabled")
except Exception as e:
    containment_actions.append(f" Failed to enable enhanced monitoring: {str(e)}")
    print(f"   Failed to enable enhanced monitoring: {str(e)}")

# Update IRIS case with containment actions
iris.update_case(case_id, {
    'containment_actions': containment_actions,
    'containment_timestamp': datetime.now().isoformat(),
    'status': 'containment_in_progress'
})

print(f"\n Containment Summary:")
print(f"  Actions Taken: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in containment_actions if a.startswith('')])}")

# Trigger Shuffle workflow for containment verification
shuffle.trigger_workflow('spearphishing_containment_verification', {
    'case_id': case_id,
    'actions_taken': containment_actions
})
print(" Triggered containment verification workflow")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication Actions
print("\n" + "=" * 60)
print("STEP 3: Eradication Actions")
print("=" * 60)

eradication_actions = []

# Terminate malicious processes via CrowdStrike
print("\n💀 Terminating malicious processes...")
for process in cs_results:
    try:
        kill_result = crowdstrike.terminate_process(process['ComputerName'], process['ProcessId'])
        if kill_result.get('success'):
            eradication_actions.append(f" Terminated process: {process['ImageFileName']} on {process['ComputerName']}")
            print(f"   Terminated process: {process['ImageFileName']} on {process['ComputerName']}")
    except Exception as e:
        eradication_actions.append(f" Failed to terminate process {process['ProcessId']}: {str(e)}")
        print(f"   Failed to terminate process {process['ProcessId']}: {str(e)}")

# Remove malicious files and attachments
print("\n🗑 Removing malicious files and attachments...")
file_removal_query = """
index=* sourcetype=file_access
| where user IN (detection_summary['compromised_accounts'])
| where file_path =~ "*.exe" OR file_path =~ "*.scr" OR file_path =~ "*.bat" OR file_path =~ "*.ps1"
| stats count by host, file_path
"""

malicious_files = splunk.execute_query(file_removal_query)
for file_info in malicious_files:
    try:
        removal_result = crowdstrike.remove_file(file_info['host'], file_info['file_path'])
        if removal_result.get('success'):
            eradication_actions.append(f" Removed file: {file_info['file_path']} from {file_info['host']}")
            print(f"   Removed file: {file_info['file_path']} from {file_info['host']}")
    except Exception as e:
        eradication_actions.append(f" Failed to remove file {file_info['file_path']}: {str(e)}")
        print(f"   Failed to remove file {file_info['file_path']}: {str(e)}")

# Reset compromised credentials
print("\n🔑 Resetting compromised credentials...")
for account in detection_summary['compromised_accounts']:
    try:
        reset_result = shuffle.reset_password(account)
        if reset_result.get('success'):
            eradication_actions.append(f" Reset password for: {account}")
            print(f"   Reset password for: {account}")
    except Exception as e:
        eradication_actions.append(f" Failed to reset password for {account}: {str(e)}")
        print(f"   Failed to reset password for {account}: {str(e)}")

# Clear browser caches and temporary files
print("\n Clearing browser caches and temporary files...")
for user in detection_summary['affected_users']:
    try:
        cache_clear_result = shuffle.clear_user_cache(user)
        if cache_clear_result.get('success'):
            eradication_actions.append(f" Cleared cache for user: {user}")
            print(f"   Cleared cache for user: {user}")
    except Exception as e:
        eradication_actions.append(f" Failed to clear cache for {user}: {str(e)}")
        print(f"   Failed to clear cache for {user}: {str(e)}")

# Revoke suspicious sessions
print("\n🚪 Revoking suspicious sessions...")
session_query = """
index=* sourcetype=authentication
| where user IN (detection_summary['compromised_accounts'])
| where action="login" AND risk_score > 70
| stats count by user, session_id, src_ip
"""

suspicious_sessions = splunk.execute_query(session_query)
for session in suspicious_sessions:
    try:
        revoke_result = shuffle.revoke_session(session['user'], session['session_id'])
        if revoke_result.get('success'):
            eradication_actions.append(f" Revoked session: {session['session_id']} for {session['user']}")
            print(f"   Revoked session: {session['session_id']} for {session['user']}")
    except Exception as e:
        eradication_actions.append(f" Failed to revoke session {session['session_id']}: {str(e)}")
        print(f"   Failed to revoke session {session['session_id']}: {str(e)}")

# Verify threat removal
print("\n✅ Verifying threat removal...")
verification_results = []
for user in detection_summary['affected_users']:
    try:
        # Check for ongoing suspicious activity
        user_activity_query = f"""
        index=* sourcetype=authentication OR sourcetype=file_access
        | where user="{user}"
        | where _time > relative_time(now(), "-5m")
        | stats count
        """
        activity_results = splunk.execute_query(user_activity_query)
        if len(activity_results) == 0 or activity_results[0].get('count', 0) < 3:
            verification_results.append(f" User {user} activity normalized")
            print(f"   User {user} activity normalized")
        else:
            verification_results.append(f" User {user} still showing suspicious activity")
            print(f"   User {user} still showing suspicious activity")
    except Exception as e:
        verification_results.append(f" Verification failed for {user}: {str(e)}")
        print(f"   Verification failed for {user}: {str(e)}")

# Update IRIS case with eradication actions
iris.update_case(case_id, {
    'eradication_actions': eradication_actions,
    'verification_results': verification_results,
    'eradication_timestamp': datetime.now().isoformat(),
    'status': 'eradication_complete'
})

print(f"\n Eradication Summary:")
print(f"  Actions Taken: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Verification Results: {len([r for r in verification_results if r.startswith('')])} passed")

# Share indicators with MISP
if len(spearphishing_indicators) > 0:
    misp.share_indicators(spearphishing_indicators, case_id)
    print(" Shared spearphishing indicators with MISP community")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery Actions
print("\n" + "=" * 60)
print("STEP 4: Recovery Actions")
print("=" * 60)

recovery_actions = []

# Re-enable accounts with new credentials
print("\n👤 Re-enabling accounts with new credentials...")
for account in detection_summary['compromised_accounts']:
    try:
        reenable_result = shuffle.reenable_account(account)
        if reenable_result.get('success'):
            recovery_actions.append(f" Re-enabled account: {account}")
            print(f"   Re-enabled account: {account}")
    except Exception as e:
        recovery_actions.append(f" Failed to re-enable account {account}: {str(e)}")
        print(f"   Failed to re-enable account {account}: {str(e)}")

# Restore quarantined emails (safe ones)
print("\n📧 Restoring safe quarantined emails...")
try:
    restore_email_result = shuffle.restore_safe_emails(case_id)
    if restore_email_result.get('success'):
        recovery_actions.append(f" Restored {restore_email_result.get('emails_restored', 0)} safe emails")
        print(f"   Restored {restore_email_result.get('emails_restored', 0)} safe emails")
except Exception as e:
    recovery_actions.append(f" Failed to restore emails: {str(e)}")
    print(f"   Failed to restore emails: {str(e)}")

# Re-enable isolated endpoints
print("\n🔓 Re-enabling isolated endpoints...")
for user in detection_summary['affected_users']:
    endpoint_query = f"""
    index=* sourcetype=authentication
    | where user="{user}"
    | stats latest(src_ip) as endpoint by user
    """
    endpoint_results = splunk.execute_query(endpoint_query)
    for endpoint in endpoint_results:
        try:
            reenable_result = crowdstrike.reenable_host(endpoint['endpoint'])
            if reenable_result.get('success'):
                recovery_actions.append(f" Re-enabled endpoint: {endpoint['endpoint']} ({user})")
                print(f"   Re-enabled endpoint: {endpoint['endpoint']} ({user})")
        except Exception as e:
            recovery_actions.append(f" Failed to re-enable endpoint {endpoint['endpoint']}: {str(e)}")
            print(f"   Failed to re-enable endpoint {endpoint['endpoint']}: {str(e)}")

# Validate account integrity
print("\n Validating account integrity...")
integrity_checks = []
for account in detection_summary['compromised_accounts']:
    try:
        integrity_result = shuffle.validate_account_integrity(account)
        if integrity_result.get('integrity_valid'):
            integrity_checks.append(f" Account {account} integrity valid")
            print(f"   Account {account} integrity valid")
        else:
            integrity_checks.append(f" Account {account} integrity issues: {integrity_result.get('issues', [])}")
            print(f"   Account {account} integrity issues: {integrity_result.get('issues', [])}")
    except Exception as e:
        integrity_checks.append(f" Integrity check failed for {account}: {str(e)}")
        print(f"   Integrity check failed for {account}: {str(e)}")

# Restore user access to necessary resources
print("\n🔑 Restoring user access to resources...")
for user in detection_summary['affected_users']:
    try:
        access_restore_result = shuffle.restore_user_access(user)
        if access_restore_result.get('success'):
            recovery_actions.append(f" Restored access for user: {user}")
            print(f"   Restored access for user: {user}")
    except Exception as e:
        recovery_actions.append(f" Failed to restore access for {user}: {str(e)}")
        print(f"   Failed to restore access for {user}: {str(e)}")

# Monitor for recurrence
print("\n Establishing monitoring for recurrence...")
try:
    recurrence_monitoring = {
        'users': detection_summary['affected_users'],
        'duration_days': 30,
        'alert_threshold': 'medium',
        'indicators': ['suspicious_emails', 'login_anomalies', 'file_access_patterns']
    }
    splunk.setup_recurrence_monitoring(recurrence_monitoring)
    recovery_actions.append(" Recurrence monitoring established")
    print("   Recurrence monitoring established")
except Exception as e:
    recovery_actions.append(f" Failed to establish recurrence monitoring: {str(e)}")
    print(f"   Failed to establish recurrence monitoring: {str(e)}")

# Conduct user training notification
print("\n Sending security awareness notifications...")
for user in detection_summary['affected_users']:
    try:
        training_result = shuffle.send_security_training(user, 'spearphishing_awareness')
        if training_result.get('success'):
            recovery_actions.append(f" Sent training notification to: {user}")
            print(f"   Sent training notification to: {user}")
    except Exception as e:
        recovery_actions.append(f" Failed to send training to {user}: {str(e)}")
        print(f"   Failed to send training to {user}: {str(e)}")

# Update IRIS case with recovery actions
iris.update_case(case_id, {
    'recovery_actions': recovery_actions,
    'integrity_checks': integrity_checks,
    'recovery_timestamp': datetime.now().isoformat(),
    'status': 'recovery_complete'
})

print(f"\n Recovery Summary:")
print(f"  Actions Taken: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Integrity Checks Passed: {len([c for c in integrity_checks if c.startswith('')])}")

# Trigger Shuffle workflow for recovery verification
shuffle.trigger_workflow('spearphishing_recovery_verification', {
    'case_id': case_id,
    'actions_taken': recovery_actions,
    'integrity_checks': integrity_checks
})
print(" Triggered recovery verification workflow")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []

# Generate incident report
print("\n Generating incident report...")
try:
    incident_report = {
        'case_id': case_id,
        'title': 'Internal Spearphishing Incident Response Report',
        'timeline': {
            'detection': detection_summary['timestamp'],
            'containment': datetime.now().isoformat(),  # Would be actual timestamp
            'eradication': datetime.now().isoformat(),   # Would be actual timestamp
            'recovery': datetime.now().isoformat(),      # Would be actual timestamp
            'closure': datetime.now().isoformat()
        },
        'impact_assessment': {
            'affected_users': len(detection_summary['affected_users']),
            'compromised_accounts': len(detection_summary['compromised_accounts']),
            'suspicious_emails': detection_summary['indicators']['suspicious_emails'],
            'business_impact': 'HIGH',  # Would be determined by business impact analysis
            'data_exposure': 'POTENTIAL',  # Would be determined by investigation
            'lateral_movement_extent': 'LIMITED'  # Would be determined by investigation scope
        },
        'response_metrics': {
            'detection_to_containment': 'TBD',  # Would calculate actual time
            'containment_to_eradication': 'TBD',
            'total_resolution_time': 'TBD',
            'automation_level': 'HIGH'
        },
        'attack_characteristics': {
            'attack_vector': 'Internal Spearphishing',
            'email_campaigns': detection_summary['indicators']['suspicious_emails'],
            'compromised_credentials': detection_summary['indicators']['login_anomalies'],
            'lateral_movement_indicators': detection_summary['indicators']['file_access_patterns'],
            'insider_threat_level': 'HIGH'
        },
        'lessons_learned': [
            'Internal spearphishing campaigns can be highly effective',
            'Automated email quarantine prevented further compromise',
            'User monitoring and anomaly detection were crucial',
            'Account isolation and credential reset contained the breach',
            'MISP integration provided valuable threat context'
        ]
    }

    report_id = iris.generate_report(case_id, incident_report)
    post_incident_actions.append(f" Generated incident report: {report_id}")
    print(f"   Generated incident report: {report_id}")

except Exception as e:
    post_incident_actions.append(f" Failed to generate report: {str(e)}")
    print(f"   Failed to generate report: {str(e)}")

# Document lessons learned
print("\n Documenting lessons learned...")
lessons_learned = [
    "Implement advanced email filtering for internal communications",
    "Regular security awareness training for all users",
    "Multi-factor authentication for all accounts",
    "Continuous monitoring of user behavior and access patterns",
    "Automated response for suspicious internal activities",
    "Regular credential audits and rotation policies"
]

try:
    iris.add_lessons_learned(case_id, lessons_learned)
    post_incident_actions.append(" Documented lessons learned")
    print("   Documented lessons learned")
except Exception as e:
    post_incident_actions.append(f" Failed to document lessons learned: {str(e)}")
    print(f"   Failed to document lessons learned: {str(e)}")

# Implement preventive measures
print("\n Implementing preventive measures...")
preventive_measures = []

try:
    # Update email security policies
    policy_updates = {
        'internal_email_filtering': 'strict',
        'attachment_scanning': True,
        'url_protection': True,
        'user_training_frequency': 'quarterly'
    }
    shuffle.update_security_policies(policy_updates)
    preventive_measures.append(" Updated email security policies")
    print("   Updated email security policies")

    # Enhance monitoring rules
    monitoring_updates = {
        'internal_communication_monitoring': True,
        'user_behavior_analytics': True,
        'automated_anomaly_detection': True
    }
    splunk.update_monitoring_rules(monitoring_updates)
    preventive_measures.append(" Enhanced monitoring rules")
    print("   Enhanced monitoring rules")

    # Update threat intelligence feeds
    misp.update_feeds(['spearphishing', 'internal-threats', 'social-engineering'])
    preventive_measures.append(" Updated threat intelligence feeds")
    print("   Updated threat intelligence feeds")

except Exception as e:
    preventive_measures.append(f" Failed to implement preventive measures: {str(e)}")
    print(f"   Failed to implement preventive measures: {str(e)}")

# Conduct post-incident review
print("\n Conducting post-incident review...")
try:
    review_findings = {
        'effectiveness_rating': 'HIGH',
        'areas_for_improvement': [
            'Earlier detection of internal spearphishing',
            'Better user training programs',
            'Enhanced email security controls'
        ],
        'recommendations': [
            'Implement AI-based email analysis',
            'Regular phishing simulation exercises',
            'Advanced user and entity behavior analytics',
            'Zero-trust architecture implementation',
            'Automated incident response playbooks'
        ]
    }

    iris.conduct_post_incident_review(case_id, review_findings)
    post_incident_actions.append(" Conducted post-incident review")
    print("   Conducted post-incident review")

except Exception as e:
    post_incident_actions.append(f" Failed to conduct review: {str(e)}")
    print(f"   Failed to conduct review: {str(e)}")

# Close the incident case
print("\n Closing incident case...")
try:
    closure_data = {
        'closure_reason': 'Internal spearphishing campaign contained - accounts secured, monitoring enhanced',
        'closure_timestamp': datetime.now().isoformat(),
        'final_status': 'CLOSED',
        'follow_up_required': False,
        'reopen_criteria': 'Detection of similar internal spearphishing activity'
    }

    iris.close_case(case_id, closure_data)
    post_incident_actions.append(" Closed incident case")
    print("   Closed incident case")

except Exception as e:
    post_incident_actions.append(f" Failed to close case: {str(e)}")
    print(f"   Failed to close case: {str(e)}")

# Final summary
print(f"\n Post-Incident Summary:")
print(f"  Case ID: {case_id}")
print(f"  Actions Completed: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Preventive Measures: {len(preventive_measures)}")
print(f"  Lessons Learned: {len(lessons_learned)}")

print("\n Incident Response Complete")
print("=" * 60)
print("Internal spearphishing incident successfully contained and resolved.")
print("Accounts secured, monitoring enhanced, and preventive measures implemented.")
print("=" * 60)


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
